<a href="https://colab.research.google.com/github/AI-is-out-there/2026-REA-Diploma-Practical-Part/blob/dev/FireCrawl-CR-site-parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install the latest version of Firecrawl
!pip install firecrawl-py --upgrade -q

from firecrawl import Firecrawl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.1/248.1 kB 7.3 MB/s eta 0:00:00


In [2]:
API_KEY = 'fc-7f29fdf62edf482c9bd1a65cbcb4eb78'

In [3]:
app = Firecrawl(api_key=API_KEY)

def search_and_extract_with_ai(keyword):
    """
    Uses Firecrawl's new AI 'Interact' feature to control a real browser
    using natural language prompts. This bypasses all complex JavaScript
    and anti-bot protections.
    """
    print(f"🔍 Starting Firecrawl AI session for: '{keyword}'...")

    try:
        # 1. Initial scrape to start the browser session and get a scrape_id
        print("🌐 Opening the clinical guidelines registry...")
        result = app.scrape("https://cr.minzdrav.gov.ru/clin-rec", formats=["markdown"])

        # Extract the session ID (handle both object and dict returns just in case)
        if hasattr(result, 'metadata') and hasattr(result.metadata, 'scrape_id'):
            scrape_id = result.metadata.scrape_id
        elif isinstance(result, dict) and 'metadata' in result:
            scrape_id = result['metadata'].get('scrape_id') or result['metadata'].get('scrapeId')
        else:
            print("❌ Could not find scrape_id in the response.")
            return None

        print(f"✅ Browser session started. ID: {scrape_id}")

        # 2. AI Action: Type the keyword and search
        print(f"🔎 AI Agent is typing '{keyword}' into the search box...")
        app.interact(
            scrape_id,
            prompt=f"Find the search input field on this page, type '{keyword}' into it, and submit the search (press Enter or click the search button)."
        )

        # 3. AI Action: Click the first result
        print("📄 AI Agent is clicking the first search result...")
        app.interact(
            scrape_id,
            prompt="Wait for the search results to load. Then, click on the very first link in the search results list to open the Clinical Recommendation page."
        )

        # 4. AI Action: Navigate to Appendix B and extract the text
        print("📑 AI Agent is finding and extracting 'Doctor's Action Algorithms' (Appendix B)...")
        response = app.interact(
            scrape_id,
            prompt=(
                "Find the section titled 'Приложение Б' or 'Алгоритмы действий врача' (Doctor's Action Algorithms). "
                "Click on it to expand it if it is collapsed. "
                "Then, extract and return the full text content of this specific section. "
                "Do not summarize it, return the exact text."
            )
        )

        # The AI agent returns the extracted text in the 'output' field
        extracted_text = response.output
        print("✅ Successfully extracted the Doctor's Actions Algorithm!")

        # 5. Clean up the browser session to save credits
        app.stop_interaction(scrape_id)
        print("🧹 Browser session closed.")

        return extracted_text

    except Exception as e:
        print(f"❌ An error occurred: {e}")
        return None

# --- RUN THE SCRIPT ---
search_keyword = "Прогрессирующая мышечная дистрофия"
result = search_and_extract_with_ai(search_keyword)

if result:
    print("\n" + "="*60)
    print("EXTRACTED DOCTOR ACTIONS ALGORITHM:")
    print("="*60)
    print(result)
else:
    print("Failed to extract data.")

🔍 Starting Firecrawl AI session for: 'Прогрессирующая мышечная дистрофия'...
🌐 Opening the clinical guidelines registry...
✅ Browser session started. ID: 019eff41-892e-707a-9f4e-649a2698a439
🔎 AI Agent is typing 'Прогрессирующая мышечная дистрофия' into the search box...
📄 AI Agent is clicking the first search result...
📑 AI Agent is finding and extracting 'Doctor's Action Algorithms' (Appendix B)...
✅ Successfully extracted the Doctor's Actions Algorithm!
🧹 Browser session closed.

EXTRACTED DOCTOR ACTIONS ALGORITHM:
The requested sections "Приложение Б" or "Алгоритмы действий врача" are not present on the current page. The page indicates that technical work is being performed.
